#### importing libraries

In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

In [21]:
df = pd.read_csv("C://Users//Nandhika//Documents//ANN_BankChurnPrediction//data//Churn_Modelling.csv")
df.head(3)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


##### Preprocessing the data

In [22]:
df.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)

In [6]:
df.head(3)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


In [7]:
df['Gender'].unique()

array(['Female', 'Male'], dtype=object)

categorical variables with only two features so we can use binary encoding

In [23]:
le_gender = LabelEncoder()
df['Gender'] = le_gender.fit_transform(df['Gender'])


In [11]:
df['Gender'].unique()

array([0, 1])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  int32  
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int32(1), int64(7), object(1)
memory usage: 820.4+ KB


In [8]:
df['Geography'].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [24]:
ohe_geo = OneHotEncoder()
ohe_geo_transformed = ohe_geo.fit_transform(df[['Geography']]) #fit it using the feature and save it in the transformed variable

In [25]:
ohe_geo_transformed.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [26]:
ohe_geo_df = pd.DataFrame(ohe_geo_transformed.toarray(),columns=ohe_geo.get_feature_names_out(['Geography']))
ohe_geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [27]:
# Combining everything into one dataframe
df = pd.concat([df.drop(['Geography'],axis=1),ohe_geo_df],axis=1)
df.head(3)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0


In [29]:
# Pickling the encoders 
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(le_gender,file)
with open('ohe_geography.pkl','wb') as file:
    pickle.dump(ohe_geo,file)


In [30]:
# Splitting into dependent and independent features
X = df.drop(['Exited'],axis=1)
y = df['Exited']

In [42]:
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,0.0,1.0,0.0


In [33]:
# Dividing into training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

#Scaling features
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [34]:
with open('scaler.pkl','wb') as file:
    pickle.dump(sc,file)

### ANNnnnnnnnnnnnnnn....

In [37]:
import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

In [41]:
X_train.shape[1]

12

In [38]:
#Building ANN model
model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)),
    Dense(34, activation='relu'),
    Dense(1,activation='sigmoid')
])

input_shape=(...) tells Keras: “the data coming into this layer has this many features”.
Keras will automatically create an input layer with one neuron per feature.
So the “input layer” neurons = number of features in X_train, i.e., X_train.shape[1].

Input layer (11 neurons)  -->  Dense layer 1 (64 neurons)
Each of 64 neurons takes all 11 inputs
Dense layer 2 (34 neurons)
Each neuron takes all 64 outputs from previous layer
Output layer (1 neuron)

In [39]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 34)                2210      
                                                                 
 dense_2 (Dense)             (None, 1)                 35        
                                                                 
Total params: 3077 (12.02 KB)
Trainable params: 3077 (12.02 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [43]:
## Defining the optimizers
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [44]:
## Defining some things during training
model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
## Setting up tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [46]:
# EarlyStopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [48]:
## Model Traininggggg
history = model.fit(
    X_train,y_train,validation_data = (X_test,y_test), epochs = 100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3932 - accuracy: 0.8370 - val_loss: 0.3553 - val_accuracy: 0.8530
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3548 - accuracy: 0.8555 - val_loss: 0.3458 - val_accuracy: 0.8500
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3474 - accuracy: 0.8551 - val_loss: 0.3443 - val_accuracy: 0.8590
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3411 - accuracy: 0.8615 - val_loss: 0.3459 - val_accuracy: 0.8580
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3414 - accuracy: 0.8614 - val_loss: 0.3422 - val_accuracy: 0.8605
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3372 - accuracy: 0.8611 - val_loss: 0.3601 - val_accuracy: 0.8510
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3361 - accuracy: 0.8618 - val_loss: 0.3399 - val_accuracy: 0.8605

In [49]:
# saving model file
# h5 is compatiable with keras
model.save('modelone.h5')

c:\Users\Nandhika\Documents\ANN_BankChurnPrediction\churnvenv\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [50]:
# Launching tensorboard extension
%load_ext tensorboard

In [52]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 123148), started 0:01:59 ago. (Use '!kill 123148' to kill it.)